# Prompt engineering

### Cleanup: Free Apple Silicon (MPS) Memory

This utility function ensures that any previously loaded models and tokenizers are deleted and GPU memory is cleared.

This is particularly useful when re-running sections or switching models on Apple Silicon (M1/M2) machines using `torch.mps`.

We call this cleanup before loading new models to avoid memory errors or slowdowns.

In [8]:
def cleanup_mps_memory():
    """
    Frees MPS memory by deleting global variables 'model' and 'tokenizer' if they exist.
    Useful when you want to avoid passing model/tokenizer manually.
    """
    import gc
    import torch

    for var in ['model', 'tokenizer']:
        if var in globals():
            print(f"🔹 Deleting: {var}")
            del globals()[var]

    gc.collect()
    torch.mps.empty_cache()
    print("MPS memory cleaned.")
cleanup_mps_memory()

MPS memory cleaned.


### Loading a Local Language Model for Text Generation

In this section, we're setting up a **Hugging Face Transformers** model (Phi-3 Mini by Microsoft) to use locally with PyTorch. This enables us to perform tasks like prompt completion, information extraction, and structured data generation.

#### Step-by-step Breakdown

- **Imports**:  
  We begin by importing key modules:
  - `AutoModelForCausalLM` and `AutoTokenizer` from `transformers` for loading the pretrained model and tokenizer.
  - `pipeline` from `transformers` to simplify interaction with the model.
  - `torch` to manage the computation device.

- **Apple Silicon Support (MPS)**:  
  We check if MPS (Metal Performance Shaders) is available, which provides hardware acceleration on Apple Silicon (M1/M2). If not, we default to CPU.

- **Model Loading**:  
  The model and tokenizer are loaded using the `from_pretrained` method:
  - `'microsoft/Phi-3-mini-4k-instruct'` is the selected instruction-tuned language model.
  - `torch_dtype=torch.float32` ensures MPS compatibility.
  - `trust_remote_code=True` is needed for custom model implementations from the repo.

- **Pipeline Setup**:  
  We create a text generation pipeline that wraps the model and tokenizer:
  - `return_full_text=False`: Only returns the generated output (not the prompt).
  - `max_new_tokens=500`: Limits the output to a maximum of 500 new tokens.
  - `do_sample=False`: Disables randomness, ensuring deterministic and reproducible output — important for structured tasks like extraction.


In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Set device to Apple MPS if available, fallback to CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Load tokenizer and model from Hugging Face
tokenizer = AutoTokenizer.from_pretrained('microsoft/Phi-3-mini-4k-instruct')
model = AutoModelForCausalLM.from_pretrained(
    'microsoft/Phi-3-mini-4k-instruct',
    torch_dtype=torch.float32,
    trust_remote_code=True
).to(device)

# Create a pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False,
)

/opt/miniconda3/envs/re/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.
Loading checkpoint shards: 100%|██████████| 2/2 [00:11<00:00,  5.99s/it]


### Regular Prompt: Extracting Structured Travel Preferences

In this example, we send a simple prompt to the model to extract structured information from a travel-related user message.

#### Prompt Structure

We format the input as a list of `messages`, following the chat-style format used in most instruction-tuned models:

- **System Message**:  
  Provides instruction to the model. It tells the model what to do:
  - Extract structured travel preferences.
  - Return the information in **JSON format**.
  - Use the following fields: `country`, `city`, `travel_type`, `duration`, `travel_dates`, `budget`, `companions`, `interests`.
  - If any field is missing in the message, its value should be `null`.

- **User Message**:  
  Represents the input text we want to process. It describes a trip to Italy, with preferences for travel style, companions, budget, etc.

#### Running the Pipeline

We pass the `messages` list to the `pipe`, which uses the loaded model to generate output.

#### Key Takeway:
Even a simple prompt (without examples or extra context) can perform very well on structured extraction tasks — especially when the model is instruction-tuned and the task is clearly defined.


In [ ]:
messages = [
    {
        'role': 'system', 
        'content': """
Identify and extract the user's stated travel preferences and requirements from the message.
Return the extracted information in valid JSON format with the following fields:
country, city, travel_type, duration, travel_dates, budget, companions, interests.
If a field is not mentioned in the user's message, set its value to null.
"""
    },
    {
        'role': 'user', 
        'content': """
I want to take a relaxing trip to Italy in July 2025, preferably near the coast. I'd like to go with my partner, ideally for around 10 days. 
We're food lovers, so great local cuisine is a must. I'd like to spend around 2000-3000 EUR in total.\n
"""
    }
]

In [11]:
output = pipe(messages)

You are not running the flash-attention implementation, expect numerical differences.


In [12]:
# good output!
print(output[0]['generated_text'])

 ```json

{

  "country": "Italy",

  "city": null,

  "travel_type": "Relaxing",

  "duration": "10 days",

  "travel_dates": "July 2025",

  "budget": "2000-3000 EUR",

  "companions": "Partner",

  "interests": "Local cuisine"

}

```


### Behind the Scenes: How `transformers.pipeline` Formats Prompts

When using the `pipeline` API for chat-based models (like `Phi-3`, `LLaMA`, or `Mistral`), Hugging Face uses an **internal prompt template** to format the `messages` list into a single string prompt.

This helps the model understand **who is speaking**, **what the system instructions are**, and **what needs to be completed**.

#### Explanation
- <|system|>: Marks the beginning of system-level instructions.
- <|user|>: Marks the start of the user’s message (the query).
- <|end|>: Separates sections.
- <|endoftext|>: Signals the end of the prompt for the model.

These tags are model-specific. For example:
- phi-3 uses <|system|>, <|user|>, etc.
- OpenAI models like GPT don’t require such tags explicitly — their API handles this automatically.
- For many other Hugging Face models, formatting differs — always consult the tokenizer’s docs or check apply_chat_template().

#### Why It Matters
Understanding how the pipeline formats prompts:
- Helps when debugging or manually building prompts.
- Lets you fine-tune prompt templates for better model alignment.
- Makes it easier to port prompts between APIs or models.

In [13]:
# under the hood, transformers.pipeline first converts our message in a specific prompt template:
prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False)
print(prompt)

<|system|>

Identify and extract the user's stated travel preferences and requirements from the message.
Return the extracted information in valid JSON format with the following fields:
country, city, travel_type, duration, travel_dates, budget, companions, interests.
If a field is not mentioned in the user's message, set its value to null.
<|end|>
<|user|>

I want to take a relaxing trip to Italy in July 2025, preferably near the coast. I'd like to go with my partner, ideally for around 10 days. 
We're food lovers, so great local cuisine is a must. I'd like to spend around 2000-3000 EUR in total.

<|end|>
<|endoftext|>


### Controlling Model Output with `temperature` and `top_p`

When working with language models, we often want to **balance creativity and accuracy**. The two most common parameters to control this are:

---

#### `temperature`

- Controls how **random** or **confident** the model is when choosing the next word.
- **Higher values (e.g., 1.0 or above)** make output more diverse and creative.
- **Lower values (e.g., 0.1 or 0.2)** make the model more deterministic and focused on the most likely tokens.

---

#### `top_p` (a.k.a. nucleus sampling)

- Limits token selection to the smallest set of top probabilities whose cumulative probability is ≥ `p`.
- If `top_p = 1`, sampling is unrestricted.
- If `top_p = 0.1`, only the top 10% of probability mass is considered.

Together, `temperature` and `top_p` help fine-tune model behavior:

---

### Example: High Diversity (temperature=1, top_p=1)
**Notice**: The outputs vary each time — good for creative writing, not ideal for structured extraction.

In [14]:
output = pipe(messages, do_sample=True, temperature=1, top_p=1)
print(output[0]['generated_text'])

 ```json

{

  "country": "Italy",

  "city": null,

  "travel_type": "Camping",

  "duration": 10,

  "travel_dates": "July 2025",

  "budget": 2000,

  "companions": "Partner",

  "interests": "Great local cuisine"

}

```


In [15]:
output = pipe(messages, do_sample=True, temperature=1, top_p=1)
print(output[0]['generated_text'])

 ```json

{

  "country": "Italy",

  "city": null,

  "travel_type": "relax",

  "duration": "10",

  "travel_dates": "July 2025",

  "budget": "2000-3000",

  "companions": "partner",

  "interests": "local cuisine"

}

```


#### Example: High Precision (temperature=0.1, top_p=0.1)
**Notice**: The outputs don't vary each time — good for structured extraction.
- Every time you run the LLM, you can expect the same result - better for production systems expecting consistency. 
- temperature=0.1: keeps token choice very focused on the most probable next word.
- top_p=0.1: only samples from a very narrow set of likely tokens.

In [ ]:
output = pipe(messages, do_sample=True, temperature=0.1, top_p=0.1)
print(output[0]['generated_text'])

 ```json

{

  "country": "Italy",

  "city": null,

  "travel_type": "Relaxing",

  "duration": "10 days",

  "travel_dates": "July 2025",

  "budget": "2000-3000 EUR",

  "companions": "Partner",

  "interests": "Local cuisine"

}

```


In [17]:
output = pipe(messages, do_sample=True, temperature=0.1, top_p=0.1)
print(output[0]['generated_text'])

 ```json

{

  "country": "Italy",

  "city": null,

  "travel_type": "Relaxing",

  "duration": "10 days",

  "travel_dates": "July 2025",

  "budget": "2000-3000 EUR",

  "companions": "Partner",

  "interests": "Local cuisine"

}

```


 ### Advanced Prompt Engineering

Prompt engineering isn't just about writing instructions — it's about carefully designing the **structure** and **context** of the prompt to get high-quality, structured output. This section demonstrates how to include:

- ✅ **Persona** (who is answering)
- ✅ **Instruction** (what to do)
- ✅ **Context** (why the task matters)
- ✅ **Format** (what the output should look like)
- ✅ **Audience** (who will use the output)
- ✅ **Tone** (how the output should sound)
- ✅ **User data** (the actual task)

In [18]:
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

In [19]:
import openai

# Create client
client = openai.OpenAI(api_key=api_key)

In [ ]:
persona = "You are a precise and helpful assistant that extracts structured travel preferences from user messages.\n"

instruction = "Identify and extract the user's stated travel preferences and requirements from the message.\n"

context = """
These preferences will be used to help match the user with personalized travel options and destinations.
Your output must be accurate and in a structured format to support automated processing.\n
"""

data_format = """Return the extracted information in valid JSON format with the following fields:

{
  "country": ...,        // e.g. "Italy", "Japan", "Mexico"
                         // User may say: "vacation in Italy", "somewhere in Japan", "traveling to Mexico"

  "city": ...,           // e.g. "Rome", "Tokyo", "Cancún"
                         // User may say: "near Rome", "around Tokyo", "maybe Cancún"

  "travel_type": ...,    // e.g. "beach", "adventure", "cultural", "relaxation", "city break"
                         // User may say: "looking for an adventure", "want to relax", "love city breaks"

  "duration": ...,       // e.g. "1 week", "7-10 days", "a weekend", "2 weeks"
                         // User may say: "a week-long trip", "staying 10 days", "just a weekend"

  "travel_dates": ...,   // e.g. "June 2024", "early August", "next spring"
                         // User may say: "in June", "around Christmas", "late summer"

  "budget": ...,         // e.g. 2000, "1500-2500", "max 3k EUR"
                         // User may say: "up to 2000", "budget 2-3 thousand", "not more than 3k"

  "companions": ...,     // e.g. "solo", "family", "couple", "friends"
                         // User may say: "traveling with kids", "a romantic trip", "going with friends"

  "interests": ...       // List of keywords: e.g. ["museums", "hiking", "food", "beaches"]
                         // User may say: "I love history and food", "want to hike and swim", "interested in museums and art"
}

If a field is not mentioned in the user's message, set its value to null.
"""

audience = "The output will be used by a travel recommendation engine to match the user with suitable destinations and activities.\n"

tone = "Maintain a clear, formal, and professional tone. Avoid assumptions. Do not infer extra preferences not mentioned explicitly.\n"

user_message = """
I want to take a relaxing trip to Italy in July 2025, preferably near the coast. I'd like to go with my partner, ideally for around 10 days. 
We're food lovers, so great local cuisine is a must. I'd like to spend around 2000-3000 EUR in total.\n
"""

data = f"""
User message: {user_message}
json output: 
"""

In [21]:
query = persona + instruction + context + data_format + audience + tone + data

In [22]:
messages = [{'role': 'user', 'content': query}]

In [23]:
chat_completion = client.chat.completions.create(
      messages=messages,
      model="gpt-3.5-turbo",
      temperature=0
    )

print(chat_completion.choices[0].message.content)

{
  "country": "Italy",
  "city": null,
  "travel_type": "relaxation",
  "duration": "around 10 days",
  "travel_dates": "next summer",
  "budget": "2000-3000 EUR",
  "companions": "couple",
  "interests": ["food"]
}


### Reasoning with Chain-of-Thought (CoT) and Tree-of-Thought (ToT)

While regular prompts often produce correct answers, **reasoning prompts** like Chain-of-Thought (CoT) and Tree-of-Thought (ToT) can improve performance on more complex problems — especially those requiring multi-step logic or error-checking.

---

In [ ]:
question = """
Julia is planning a trip to Europe and has a budget of €3,000.

She wants to visit three countries: Italy, France, and Germany.

Flights between each country cost €200.

Hotels in each country cost €100 per night.

She plans to stay 3 nights in each country.

Julia wants to keep at least €500 aside for food and activities.

Question: Can Julia afford the trip as planned within her €3,000 budget?

"""

Expected Output: 
- Flights: 3 countries x €200 = €600
- Hotels: 3 countries x €100 x 3 nights = €900
- Food & activities: €500
- Total: €1,800


In [49]:
regular_messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": question}
]


chat_completion = client.chat.completions.create(
      messages=regular_messages,
      model="gpt-3.5-turbo",
      temperature=0
    )

print(chat_completion.choices[0].message.content)

Let's calculate the total cost of Julia's trip:

Flights:
3 countries x €200 = €600

Hotels:
3 countries x €100 per night x 3 nights = €900

Food and activities:
€500

Total cost:
Flights (€600) + Hotels (€900) + Food and activities (€500) = €2,000

Since the total cost of the trip is €2,000, Julia can afford the trip as planned within her €3,000 budget. She will have €1,000 left from her budget after the trip.


In [ ]:
tot_messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": f"""
Imagine three travel agents are planning Julias trip.

Each of them estimates the total cost based on flights, accommodation, and activities.

They then compare their estimates and correct each other's mistakes.

Can they agree on whether the trip fits Julia's €3,000 budget?
Question:  
{question}
"""}
]


chat_completion = client.chat.completions.create(
      messages=tot_messages,
      model="gpt-3.5-turbo",
      temperature=0
    )

print(chat_completion.choices[0].message.content)

Let's break down the estimated costs for Julia's trip:

Flights:
Julia will need to take two flights between the three countries, so the total cost for flights will be 2 flights x €200 = €400.

Accommodation:
Julia plans to stay 3 nights in each country, so the total cost for accommodation will be 3 nights x €100 per night x 3 countries = €900.

Food and activities:
Julia wants to keep at least €500 aside for food and activities.

Total estimated cost:
Flights (€400) + Accommodation (€900) + Food and activities (€500) = €1,800.

Based on the estimates provided, the total estimated cost of Julia's trip is €1,800, which is well within her €3,000 budget. Therefore, Julia can afford the trip as planned within her budget.


In [52]:
cot_messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": f"""
{question}

Let's go through this step by step:

1. How many flights are needed, and how much will they cost?
2. How many hotel nights in total, and what will that cost?
3. Add a food/activity budget.
4. Compare total with the €3,000 budget.
5. Can she afford it?
"""}
]


chat_completion = client.chat.completions.create(
      messages=cot_messages,
      model="gpt-3.5-turbo",
      temperature=0
    )

print(chat_completion.choices[0].message.content)

1. Julia will need 2 flights between the 3 countries (Italy to France, France to Germany). The total cost for flights will be 2 flights x €200 = €400.

2. Julia plans to stay 3 nights in each country, so she will need a total of 3 nights x 3 countries = 9 hotel nights. The total cost for hotels will be 9 nights x €100 = €900.

3. Julia wants to keep at least €500 aside for food and activities.

4. Adding up the costs:
Flights: €400
Hotels: €900
Food/Activities: €500

Total cost = €400 + €900 + €500 = €1,800

5. Julia's total cost is €1,800, which is well within her budget of €3,000. Therefore, Julia can afford the trip as planned within her budget.


### 📌 Key Takeaways

| Prompt Type | Strength                          | Best For                          |
|-------------|------------------------------------|------------------------------------|
| Regular     | Fast, simple                      | Straightforward Q&A                |
| CoT         | Step-by-step logic                | Math, logic, budgeting problems    |
| ToT         | Multiple perspectives + checking  | Debate, self-correction tasks      |


### Controlling output - Zero-shot vs One-shot vs Few-shot Prompting

| Prompt Type | Description | Example Use Case | Strengths | Weaknesses |
|-------------|-------------|------------------|-----------|------------|
| **Zero-shot** | No examples given. The model is asked to perform the task directly with only instructions. | "Extract travel preferences from this message and return JSON." | Fast, compact, works well with simple tasks. | May miss nuances, especially in formatting or uncommon phrasing. |
| **One-shot** | One example is provided before asking the model to perform the task on a new input. | Show a formatted extraction example, then ask to do the same for a new message. | Helps steer the model, especially on formatting and structure. | Still limited in capturing edge cases or variety. |
| **Few-shot** | Two or more examples provided before the actual task. | Show 2+ sample messages and outputs, then ask for extraction from a new message. | More robust understanding of task structure and style. | Prompt gets longer; may require trimming if using smaller models. |

---

#### When to Use What?

- **Zero-shot**: Great for simple, well-defined tasks and tight latency constraints.
- **One-shot**: Ideal when format compliance is essential (e.g., exact JSON structure).
- **Few-shot**: Useful when user messages vary a lot or when richer context/examples improve reliability.

---
**By adjusting the number of examples, we can steer the model to behave more predictably, especially when user inputs vary significantly in phrasing or structure.**

In [58]:
user_message = """
I want to go to Italy in June for about 10 days. I'd like to see historical places and maybe spend a few days by the sea. I prefer cities like Rome or Florence. My budget is around 2000 euros including flights.
"""

In [59]:
zero_shot_prompt = [
    {"role": "user", "content": f"""
Extract the travel preferences from this message and return them in JSON format: {user_message}
"""}
]


chat_completion = client.chat.completions.create(
      messages=zero_shot_prompt,
      model="gpt-3.5-turbo",
      temperature=0
    )

print(chat_completion.choices[0].message.content)


{
  "destination": "Italy",
  "month": "June",
  "duration": "10 days",
  "interests": ["historical places", "spending time by the sea"],
  "preferred_cities": ["Rome", "Florence"],
  "budget": "2000 euros including flights"
}


In [57]:
zero_shot_prompt_format = [
    {"role": "user", "content": f"""
Extract travel preferences in the following JSON format:

{{
  "destination": ...,
  "travel_month": ...,
  "duration_days": ...,
  "interests": [...],
  "preferred_cities": [...],
  "budget_eur": ...,
  "includes_flights": true/false
}}

Now extract from this message: 
{user_message}
"""}
]


chat_completion = client.chat.completions.create(
      messages=zero_shot_prompt_format,
      model="gpt-3.5-turbo",
      temperature=0
    )

print(chat_completion.choices[0].message.content)

{
  "destination": "Italy",
  "travel_month": "June",
  "duration_days": 10,
  "interests": ["historical places", "sea"],
  "preferred_cities": ["Rome", "Florence"],
  "budget_eur": 2000,
  "includes_flights": true
}


In [62]:
one_shot_prompt = [
    {"role": "user", "content": f"""
Extract travel preferences in structured JSON format.

Example:
Message: "I want to go to Japan in April. I'll be there for 7 days. I'm interested in cherry blossoms and local culture. Tokyo and Kyoto are my top choices. Budget is 2500 EUR, excluding flights."

Output:
{{
  "destination": "Japan",
  "travel_month": "April",
  "duration_days": 7,
  "interests": ["cherry blossoms", "local culture"],
  "preferred_cities": ["Tokyo", "Kyoto"],
  "budget_eur": 2500,
  "includes_flights": false
}}

Now extract from this message:
{user_message}
"""}
]


chat_completion = client.chat.completions.create(
      messages=one_shot_prompt,
      model="gpt-3.5-turbo",
      temperature=0
    )

print(chat_completion.choices[0].message.content)

{
  "destination": "Italy",
  "travel_month": "June",
  "duration_days": 10,
  "interests": ["historical places", "spend a few days by the sea"],
  "preferred_cities": ["Rome", "Florence"],
  "budget_eur": 2000,
  "includes_flights": true
}


In [63]:
few_shot_prompt = [
    {"role": "user", "content": f"""
Extract travel preferences in structured JSON format.

Example 1:
Message: "I'm planning a short trip to France in May for about 5 days. I love art museums and local food. I'd like to visit Paris and maybe Lyon. Budget is 1500 EUR, excluding flights."

Output:
{{
  "destination": "France",
  "travel_month": "May",
  "duration_days": 5,
  "interests": ["art museums", "local food"],
  "preferred_cities": ["Paris", "Lyon"],
  "budget_eur": 1500,
  "includes_flights": false
}}

Example 2:
Message: "Looking to travel around Spain for 2 weeks in September. I want beaches and nightlife. Thinking Barcelona and Ibiza. Budget is 1800 EUR, including flights."

Output:
{{
  "destination": "Spain",
  "travel_month": "September",
  "duration_days": 14,
  "interests": ["beaches", "nightlife"],
  "preferred_cities": ["Barcelona", "Ibiza"],
  "budget_eur": 1800,
  "includes_flights": true
}}

Now extract from this message:
{user_message}
"""}
]


chat_completion = client.chat.completions.create(
      messages=few_shot_prompt,
      model="gpt-3.5-turbo",
      temperature=0
    )

print(chat_completion.choices[0].message.content)

{
  "destination": "Italy",
  "travel_month": "June",
  "duration_days": 10,
  "interests": ["historical places", "sea"],
  "preferred_cities": ["Rome", "Florence"],
  "budget_eur": 2000,
  "includes_flights": true
}


### 🎁 Bonus: Structured Outputs with `text_format` (OpenAI + Pydantic)

OpenAI now supports **structured response parsing** using `text_format` and Pydantic models. This allows LLMs to return well-typed, validated outputs directly mapped to Python classes.

Using `client.responses.parse()`, we guide the LLM to reason through a math problem step by step and parse the output into a structured format.

Benefits:
- Clear reasoning flow
- Easy to validate or visualize
- Helpful for tutoring, QA, or multi-step logic tasks

#### Example 1: Math Reasoning — Step-by-Step Output

In [129]:
from pydantic import BaseModel

class Step(BaseModel):
    explanation: str
    output: str

class MathReasoning(BaseModel):
    steps: list[Step]
    final_answer: str

response = client.responses.parse(
    model="gpt-4o-2024-08-06",
    input=[
        {
            "role": "system",
            "content": "You are a helpful math tutor. Guide the user through the solution step by step.",
        },
        {"role": "user", "content": question},
    ],
    text_format=MathReasoning,
)

math_reasoning = response.output_parsed

In [130]:
math_reasoning.steps

[Step(explanation='Calculate the total cost of flights between the three countries. Julia needs flights between each country, so there are flights from Italy to France and from France to Germany.', output='Flights cost = 2 flights * €200 = €400'),
 Step(explanation='Calculate the cost of hotels. Julia plans to stay 3 nights in each country, which means a total of 3 nights for 3 countries.', output='Hotels cost = 3 nights * 3 countries * €100 = €900'),
 Step(explanation='Calculate the total expenditure on flights and hotels combined.', output='Total cost for flights and hotels = €400 (flights) + €900 (hotels) = €1300'),
 Step(explanation='Subtract the amount Julia wants to keep for food and activities from her total budget to see how much she can spend on flights and hotels.', output='Amount available for flights and hotels = €3,000 - €500 = €2,500'),
 Step(explanation='Compare the total cost for flights and hotels with the amount available for them to see if she can afford the trip.', 

In [131]:
math_reasoning.final_answer

'Yes, Julia can afford the trip.'

####  Example 2: Extracting Travel Plans from User Text

In [126]:
user_message = """
I want to go to Italy in June for about 10 days. I'd like to see historical places and maybe spend a few days by the sea. I prefer cities like Rome or Florence. My budget is around 2000 eur including flights.
"""

In [ ]:
from pydantic import BaseModel
from typing import Optional
from enum import Enum

class Currency(str, Enum):
    eur = 'eur'
    pln = 'pln'
    usd = 'usd'

class TravelPlansExtraction(BaseModel):
    destination_country: list[str]
    destination_city: list[str]
    travel_month: list[str]
    duration_days: int
    interests: list[str]
    budget: float
    budget_currency: Optional[Currency]
    includes_flights: bool
    companions: list[str]

try:
    response = client.responses.parse(
        model="gpt-4o-2024-08-06",
        input=[
            {
                "role": "system",
                "content": "You are an expert at structured data extraction at a travel agency. You will be given unstructured text specifying user travel dreams and plan and you should convert it into the given structure.",
            },
            {"role": "user", "content": user_message},
        ],
        text_format=TravelPlansExtraction,
    )
except Exception as e:
    pass

In [128]:
travel_plans = response.output_parsed
travel_plans

TravelPlansExtraction(destination_country=['Italy'], destination_city=['Rome', 'Florence'], travel_month=['June'], duration_days=10, interests=['historical places', 'seaside'], budget=2000.0, budget_currency=<Currency.usd: 'usd'>, includes_flights=True, companions=[])